# 差分発現トップN解析実行

**対応記事**: [article-08c-differential-topn-analysis.md](../blog/article-08c-differential-topn-analysis.md) — トップN解析実行  
**実行順序**: 8c番目  
**所要時間**: 約15分

---

## このNotebookで行うこと

前回（notebook_08b）では1,055個の有意差タンパク質全体でのクラスタリングとPCAを実行しました。この記事では、最も有意差の大きいTop 50/100/200タンパク質だけでも群分離が可能かを検証するためのTop N解析を実行します。

- 変化量に基づくTop Nタンパク質の選択
- Up/Down方向のバランス取れた選択アルゴリズム
- 段階的解析（50→100→200）による性能評価
- 各段階でのクラスタリングとPCA可視化

**⚠️ 注意**: このNotebookを実行する前に、差分発現解析の結果が必要です。

## 前提条件

- [notebook_08b_differential_clustering.ipynb](./notebook_08b_differential_clustering.ipynb) が完了していること
- 差分発現タンパク質データ（`differential_proteins.csv`）が生成済み
- 前処理済みデータが利用可能であること
- Python環境が適切に設定されていること

## 1. ライブラリと設定

In [ ]:
import numpy as np             # 数値計算ライブラリ（配列操作・数学関数に使用）
import pandas as pd            # データフレーム操作ライブラリ（表形式データの読み込み・加工に使用）
import matplotlib.pyplot as plt  # グラフ描画ライブラリ（PCAプロット等の作成に使用）
from matplotlib.patches import Ellipse  # 楕円パッチ（PCAプロットに95%信頼楕円を描画するため）
import matplotlib.transforms as transforms  # 座標変換（楕円の回転・拡縮・移動に使用）
import seaborn as sns          # 統計的可視化ライブラリ（ヒートマップ等の高水準プロットに使用）
from sklearn.decomposition import PCA  # 主成分分析（PCA）クラス（次元削減による群分離の可視化に使用）
import os                     # OS操作ライブラリ（ディレクトリ作成に使用）

# Jupyter notebook での図のインライン表示設定
%matplotlib inline

In [ ]:
# --- パス ---
RESULTS  = "../results"              # 解析結果の保存先ディレクトリへのパス
FIG_DIR  = f"{RESULTS}/figures"      # 図の保存先ディレクトリへのパス
TABLE_DIR = f"{RESULTS}/tables"      # テーブル（CSV等）の保存先ディレクトリへのパス

# ディレクトリが存在しない場合は作成
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

print(f"結果保存先: {RESULTS}")
print(f"図保存先: {FIG_DIR}")
print(f"テーブル保存先: {TABLE_DIR}")

In [ ]:
# --- カラー ---
NORMAL, TUMOR = "#3498DB", "#E74C3C"  # Normal群を青、Tumor群を赤で表示する色コード
UP, DOWN, NS  = "#E74C3C", "#3498DB", "#CCCCCC"  # 上昇(赤)・低下(青)・非有意(灰)の色コード
# 条件名から色への対応辞書（グラフ描画時に群ごとの色を自動で割り当てるため）
COND_MAP = {"Normal": NORMAL, "Tumor": TUMOR}

print(f"色設定完了:")
print(f"Normal群: {NORMAL}, Tumor群: {TUMOR}")
print(f"Up-regulated: {UP}, Down-regulated: {DOWN}")

## 2. データ読み込み

In [ ]:
# 前処理済みデータの読み込み
df = pd.read_csv(f"{RESULTS}/preprocessed_data.csv", index_col=0)  # Log2変換・正規化済みのタンパク質発現データ
result_df = pd.read_csv(f"{TABLE_DIR}/differential_proteins.csv")   # t検定結果（統計値・有意性・Log2FC含む）

print("データ読み込み完了:")
print(f"タンパク質発現データ: {df.shape}")
print(f"差分発現解析結果: {result_df.shape}")

In [ ]:
# サンプル条件の取得（サンプル名から患者IDと条件を抽出）
conditions = pd.Series({col: "Normal" if "-N" in col else "Tumor" for col in df.columns})
print(f"データ形状: {df.shape[0]} タンパク質 × {df.shape[1]} サンプル")  # 読み込んだデータの規模を確認
print(f"有意差タンパク質数: {sum(result_df['Significant'] != 'NS')}")       # 有意差ありと判定されたタンパク質数

# サンプル条件の詳細
print(f"\nサンプル条件分布:")
print(conditions.value_counts())
print(f"\n有意差タンパク質分布:")
print(result_df["Significant"].value_counts())

## 3. 信頼楕円のヘルパー関数

**【95%信頼楕円とは？】**

- **ひとことで**: データ点の95%が含まれる楕円領域を描画する関数
- **定義**: 2次元データの共分散構造に基づいて楕円の形状・向き・サイズを決定
- **どんなとき使う**: PCAプロットで各群のデータ分散範囲を視覚的に示すとき
- **読み方**: 楕円が重複しなければ群がよく分離している証拠

In [ ]:
def confidence_ellipse(x, y, ax, n_std=2.0, **kwargs):
    """95% 信頼楕円を描画する。
    
    Parameters
    ----------
    x : array-like
        x座標のデータ
    y : array-like
        y座標のデータ
    ax : matplotlib.axes.Axes
        描画先の軸オブジェクト
    n_std : float, default=2.0
        標準偏差の倍数（2.0で約95%信頼区間）
    **kwargs
        楕円の描画オプション（色、透明度等）
    
    Returns
    -------
    matplotlib.patches.Ellipse or None
        描画された楕円オブジェクト（データ不足時はNone）
    """
    # データ点が2個未満では共分散を計算できないため、描画をスキップする
    if len(x) < 2:
        return
    # xとyの共分散行列を計算する（2x2行列: 分散と共分散を含む）
    cov = np.cov(x, y)
    # ピアソン相関係数を計算する（xとyの線形関係の強さ。-1〜+1の範囲）
    pearson = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1])
    # 楕円の横半径を計算する（相関が高いほど横長になる）
    ell_rx = np.sqrt(1 + pearson)
    # 楕円の縦半径を計算する（相関が高いほど縦短になる）
    ell_ry = np.sqrt(1 - pearson)
    # 原点中心の楕円オブジェクトを作成する（**kwargsで色や透明度を受け取る）
    ellipse = Ellipse((0, 0), width=ell_rx * 2, height=ell_ry * 2, **kwargs)
    # アフィン変換を組み合わせて楕円をデータの分布に合わせる
    transf = (transforms.Affine2D()
              .rotate_deg(45)  # 45度回転（共分散の主軸方向に合わせる）
              # n_std倍の標準偏差に合わせてスケールする（n_std=2で約95%信頼区間）
              .scale(np.sqrt(cov[0, 0]) * n_std, np.sqrt(cov[1, 1]) * n_std)
              # データの重心位置に移動する
              .translate(np.mean(x), np.mean(y)))
    # 変換を楕円に適用する（ax.transDataでデータ座標系に変換）
    ellipse.set_transform(transf + ax.transData)
    # 楕円をAxesに追加して描画する
    return ax.add_patch(ellipse)

print("信頼楕円描画関数を定義しました")

## 4. Top N 解析メイン関数

**【Top N解析とは？】**

- **ひとことで**: 最も変化の大きい上位N個のタンパク質だけで群分離が可能かを検証
- **定義**: 絶対値Log2FCでランキングし、Up/Down両方向から均等に選択
- **どんなとき使う**: 実用的なバイオマーカーパネルの最小サイズを特定するとき
- **生物学的意義**: 少数の強力なマーカーで診断可能かを評価

In [ ]:
def create_topn_analysis(df, result_df, conditions):
    """Top Nタンパク質による段階的な群分離解析を実行する。

    【解析のポイント】
    1. Up/Down両方向から均等に選択（生物学的バランス確保）
    2. 絶対値Log2FCによるランキング（変化量の大きさで評価）
    3. 段階的解析（50→100→200で性能変化を評価）
    
    Parameters
    ----------
    df : pd.DataFrame
        前処理済みタンパク質発現データ（行=タンパク質、列=サンプル）
    result_df : pd.DataFrame
        差分発現解析結果（Welch's t-test結果）
    conditions : pd.Series
        各サンプルの条件（Normal/Tumor）
    """
    # 有意差タンパク質の抽出と準備
    sig = result_df[result_df["Significant"] != "NS"].copy()  # 非有意（NS）を除外
    sig["AbsLog2FC"] = sig["Log2FC"].abs()  # 絶対値Log2FC（変化量の大きさ）を計算

    # Up（腫瘍で増加）とDown（腫瘍で減少）に分離してソート
    sig_up = sig.query("Significant == 'Up'").sort_values("AbsLog2FC", ascending=False)    # 増加タンパク質を変化量順
    sig_down = sig.query("Significant == 'Down'").sort_values("AbsLog2FC", ascending=False)  # 減少タンパク質を変化量順

    print(f"有意差タンパク質の分離:")
    print(f"Up-regulated: {len(sig_up)} proteins")     # 腫瘍で増加するタンパク質数
    print(f"Down-regulated: {len(sig_down)} proteins")  # 腫瘍で減少するタンパク質数
    
    # 各方向の代表的なタンパク質を表示
    print(f"\nTop 5 Up-regulated proteins:")
    print(sig_up.head()[["Protein", "Log2FC", "P_value"]].to_string())
    print(f"\nTop 5 Down-regulated proteins:")
    print(sig_down.head()[["Protein", "Log2FC", "P_value"]].to_string())

    # Top N解析の実行（50, 100, 200段階）
    for n in [50, 100, 200]:
        print(f"\n{'='*50}")
        print(f"=== Top {n} Analysis ====")
        print(f"{'='*50}")

        # Top Nタンパク質の選択（Up/Downから均等に選択）
        # 各方向から最大n//2個まで選択し、不足分は他方向で補完
        target_per_direction = n // 2  # 各方向の目標数
        top_up = min(target_per_direction, len(sig_up))      # Up方向から選択する数
        top_down = min(target_per_direction, len(sig_down))   # Down方向から選択する数
        
        # 不足分を他方向で補完
        remaining = n - top_up - top_down
        if remaining > 0:
            if len(sig_up) > top_up:
                top_up += min(remaining, len(sig_up) - top_up)
                remaining = n - top_up - top_down
            if remaining > 0 and len(sig_down) > top_down:
                top_down += min(remaining, len(sig_down) - top_down)

        # 実際のTop Nタンパク質リストを作成
        top_proteins = np.concatenate([
            sig_up.head(top_up)["Protein"].values,    # Up方向からtop_up個
            sig_down.head(top_down)["Protein"].values  # Down方向からtop_down個
        ])

        # 選択されたタンパク質の発現データを抽出
        top_data = df.loc[df.index.isin(top_proteins)]

        print(f"Selected proteins: {len(top_data)} (Up: {top_up}, Down: {top_down})")
        print(f"実際の選択数: {len(top_proteins)}")

        # 1. 階層的クラスタリングによる群分離確認
        print(f"\n1. 階層的クラスタリング解析中...")
        
        # サンプル色の設定（Normal/Tumorで色分け）
        sample_colors = conditions.reindex(df.columns).map(COND_MAP)

        # ClusterMapの作成（階層的クラスタリング+ヒートマップ）
        g = sns.clustermap(
            top_data.T,              # 転置してサンプル×タンパク質の形にする（行がサンプル）
            method="ward",           # Ward法（群内分散を最小化するクラスタリング手法）
            cmap="RdBu_r",           # 赤青カラーマップ（赤=高発現、青=低発現）
            z_score=1,               # 列方向（タンパク質ごと）にZ-score標準化
            row_colors=sample_colors, # 行（サンプル）の色バー（Normal/Tumor識別用）
            figsize=(8, 10),         # 図のサイズ（Top N解析用に最適化）
            xticklabels=False,       # x軸ラベル（タンパク質名）は非表示（多すぎるため）
            yticklabels=True,        # y軸ラベル（サンプル名）は表示
            vmin=-2, vmax=2,         # カラーバーの範囲（Z-score標準化後の標準的範囲）
        )

        # y軸のサンプル名フォントサイズを調整
        g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=8)

        # 図を保存
        if n == 50:
            figname = "fig2c_clustering_top50.png"   # Top 50クラスタリング
        elif n == 100:
            figname = "fig2d_clustering_top100.png"  # Top 100クラスタリング
        else:
            figname = "fig2e_clustering_top200.png"  # Top 200クラスタリング

        g.savefig(f"{FIG_DIR}/{figname}", dpi=150, bbox_inches="tight")
        plt.show()
        print(f"クラスタリング図を保存: {FIG_DIR}/{figname}")

        # 2. PCA による群分離の定量評価
        print(f"\n2. PCA解析中...")
        pca = PCA(n_components=2)  # 2次元に次元削減
        scores = pca.fit_transform(top_data.T)  # PCAを実行（行=サンプル）

        # PCAプロット作成
        fig, ax = plt.subplots(figsize=(8, 6))

        # 各群を異なる色でプロット（信頼楕円付き）
        for cond, color in [("Normal", NORMAL), ("Tumor", TUMOR)]:
            mask = (conditions.reindex(df.columns) == cond).values  # 各条件のサンプルマスク
            ax.scatter(scores[mask, 0], scores[mask, 1], c=color, s=80, alpha=0.8,
                      label=cond, edgecolors="white")  # 散布図プロット

            # 95%信頼楕円の追加（群の分布範囲を視覚化）
            confidence_ellipse(scores[mask, 0], scores[mask, 1], ax,
                               facecolor=color, alpha=0.15, edgecolor=color, lw=1.5)

        # 軸ラベル・タイトル・凡例の設定
        ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")  # 第1主成分と寄与率
        ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")  # 第2主成分と寄与率
        ax.set_title(f"PCA using Top {n} Differentially Abundant Proteins")  # グラフタイトル
        ax.legend(frameon=False)  # 凡例（枠なし）
        ax.spines[["top", "right"]].set_visible(False)  # 上・右枠線を非表示
        ax.grid(True, alpha=0.3, ls="--")  # 薄いグリッド線

        # 図を保存
        if n == 50:
            pca_figname = "fig2f_pca_top50.png"     # Top 50 PCA
        elif n == 100:
            pca_figname = "fig2g_pca_top100.png"    # Top 100 PCA
        else:
            pca_figname = "fig2h_pca_top200.png"    # Top 200 PCA

        fig.savefig(f"{FIG_DIR}/{pca_figname}", dpi=150, bbox_inches="tight")
        plt.show()
        print(f"PCA図を保存: {FIG_DIR}/{pca_figname}")

        # 基本性能の表示
        print(f"\n3. 基本性能指標:")
        print(f"PC1 variance explained: {pca.explained_variance_ratio_[0]*100:.1f}%")
        print(f"PC2 variance explained: {pca.explained_variance_ratio_[1]*100:.1f}%")
        print(f"Cumulative variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%")
        
        # 群分離度の簡易評価
        normal_mask = (conditions.reindex(df.columns) == "Normal").values
        tumor_mask = (conditions.reindex(df.columns) == "Tumor").values
        
        # 各群の重心を計算
        normal_centroid = scores[normal_mask].mean(axis=0)
        tumor_centroid = scores[tumor_mask].mean(axis=0)
        
        # 群間距離
        between_distance = np.linalg.norm(normal_centroid - tumor_centroid)
        
        print(f"群間距離: {between_distance:.2f}")
        print(f"Normal群中心: PC1={normal_centroid[0]:.2f}, PC2={normal_centroid[1]:.2f}")
        print(f"Tumor群中心:  PC1={tumor_centroid[0]:.2f}, PC2={tumor_centroid[1]:.2f}")

print("Top N解析関数を定義しました")

## 5. Top N解析の実行

In [ ]:
# Top N解析の実行
print("Top N解析を開始します...")
create_topn_analysis(df, result_df, conditions)

## 6. 結果の統計的サマリー

In [ ]:
# 各Top N解析で選択されたタンパク質の統計的特徴
sig = result_df[result_df["Significant"] != "NS"].copy()
sig["AbsLog2FC"] = sig["Log2FC"].abs()

# Up/Down分離
sig_up = sig.query("Significant == 'Up'").sort_values("AbsLog2FC", ascending=False)
sig_down = sig.query("Significant == 'Down'").sort_values("AbsLog2FC", ascending=False)

print("【Top N解析のタンパク質選択統計】")
print("\n" + "="*60)

for n in [50, 100, 200]:
    target_per_direction = n // 2
    top_up = min(target_per_direction, len(sig_up))
    top_down = min(target_per_direction, len(sig_down))
    
    # 補完計算
    remaining = n - top_up - top_down
    if remaining > 0:
        if len(sig_up) > top_up:
            additional_up = min(remaining, len(sig_up) - top_up)
            top_up += additional_up
            remaining -= additional_up
        if remaining > 0 and len(sig_down) > top_down:
            top_down += min(remaining, len(sig_down) - top_down)
    
    # 選択されたタンパク質の統計
    selected_up = sig_up.head(top_up)
    selected_down = sig_down.head(top_down)
    
    print(f"\nTop {n} タンパク質:")
    print(f"  Up-regulated: {top_up}個")
    print(f"    Log2FC範囲: {selected_up['Log2FC'].min():.2f} ~ {selected_up['Log2FC'].max():.2f}")
    print(f"    p値範囲: {selected_up['P_value'].min():.2e} ~ {selected_up['P_value'].max():.2e}")
    
    print(f"  Down-regulated: {top_down}個")
    if len(selected_down) > 0:
        print(f"    Log2FC範囲: {selected_down['Log2FC'].min():.2f} ~ {selected_down['Log2FC'].max():.2f}")
        print(f"    p値範囲: {selected_down['P_value'].min():.2e} ~ {selected_down['P_value'].max():.2e}")
    else:
        print(f"    Down-regulatedタンパク質なし")
    
    print(f"  合計: {top_up + top_down}個")

## 7. 生成された図ファイルのまとめ

In [ ]:
# 生成された図ファイルの確認
figure_files = {
    "Top 50": {
        "clustering": "fig2c_clustering_top50.png",
        "pca": "fig2f_pca_top50.png"
    },
    "Top 100": {
        "clustering": "fig2d_clustering_top100.png",
        "pca": "fig2g_pca_top100.png"
    },
    "Top 200": {
        "clustering": "fig2e_clustering_top200.png",
        "pca": "fig2h_pca_top200.png"
    }
}

print("【生成された図ファイル一覧】")
print("\n" + "="*50)

for category, files in figure_files.items():
    print(f"\n{category}:")
    for analysis_type, filename in files.items():
        filepath = f"{FIG_DIR}/{filename}"
        if os.path.exists(filepath):
            size_mb = os.path.getsize(filepath) / (1024 * 1024)
            print(f"  ✅ {analysis_type.capitalize()}: {filename} ({size_mb:.2f} MB)")
        else:
            print(f"  ❌ {analysis_type.capitalize()}: {filename} (ファイルなし)")

print(f"\n全ての図は {FIG_DIR} に保存されています。")

## まとめ

このNotebookでは以下のTop N解析を実行しました：

1. **段階的解析**: 50/100/200個の段階的なタンパク質選択
2. **バランス設計**: Up/Down両方向からの均等選択
3. **系統的可視化**: 各段階でクラスタリングとPCAを実行
4. **性能評価**: PCA寄与率と群分離度の定量的評価

**主要な成果:**
- **効率的な選択**: 絶対値Log2FCに基づく生物学的に意味のあるランキング
- **バランス確保**: Up/Down両方向の変化を捉えた選択アルゴリズム
- **可視化完了**: 6枚の図（3段階×2手法）を生成
- **定量準備**: 各段階での性能指標を取得

**技術的意義:**
- 実用的なバイオマーカーパネルサイズの評価基盤を構築
- 段階的解析による性能の閾値特定
- 少数タンパク質での診断可能性の検証

**次のステップ:**
これらの結果を定量的に評価し、最適なバイオマーカーパネルサイズの決定と実用性の詳細検討を行います。

---

## Navigation

⬅️ **前回**: [notebook_08b_differential_clustering.ipynb](./notebook_08b_differential_clustering.ipynb) — 差分発現可視化  
➡️ **次回**: [notebook_08d_differential_topn_evaluation.ipynb](./notebook_08d_differential_topn_evaluation.ipynb) — トップN評価

---

*このNotebookは [article-08c-differential-topn-analysis.md](../blog/article-08c-differential-topn-analysis.md) に対応しています。*